# COMP5329 — Deep Learning

**Tutorial 7 — Sequence Modeling Architectures I: From RNN to Transformer**

**Semester 1, 2026**

### Learning Objectives
By the end of this tutorial you will be able to:
1. Explain why ordered, variable-length data (language, time series) requires **specialised** sequence architectures and why MLPs/CNNs fall short.
2. Write the vanilla RNN recurrence from memory and derive the **vanishing / exploding gradient** result at the formula level.
3. **Implement** an `LSTMCell` from scratch — input/forget/output gates, candidate cell, additive cell-state update, hidden-state output — and explain *why* the additive path defeats vanishing gradients.
4. **Implement** a `MultiHeadSelfAttention` block from scratch — Q/K/V projections, scaled dot-product with masking, head split + concat + output projection.
5. Answer exam-style short-answer questions on the $\sqrt{d_k}$ scaling, RNN-vs-attention complexity / parallelism, and permutation equivariance / positional encoding.

### Topic Coverage

Week 7 covers **sequence models from RNN to Transformer**. The full topic list (see `Week7_Self_Study.ipynb`) is:

- ✅ **Why sequence models?** — motivation and architectural evolution *(tutorial)*
- ✅ **Vanilla RNN & the vanishing-gradient problem** — recurrence equations and Jacobian product argument *(tutorial, narrative only)*
- ✅ **LSTM cell** — gates, additive cell state, from-scratch implementation *(tutorial, in-class coding)*
- 📖 **GRU** — LSTM's lighter sibling, 2 gates instead of 3 *(self-study)*
- ✅ **Self-attention & multi-head attention** — Q/K/V, $\sqrt{d_k}$ scaling, masking, from-scratch implementation *(tutorial, in-class coding)*
- ✅ **Positional encoding** — why Transformers need it, sinusoidal formula *(tutorial, briefly)*
- 📖 **Full Transformer encoder stack, language modelling, training loops** — end-to-end runs *(self-study)*

Due to time constraints, the live tutorial focuses on **two models in depth**: the **LSTM cell** (the gating idea that rescued recurrent networks) and the **Multi-Head Self-Attention block** (the mechanism that replaced recurrence entirely). The vanilla RNN, GRU and the full Transformer training loop are left to the self-study notebook.

The live session is organised into three parts: **Part A** — tutor review, **Part B** — in-class coding exercise, **Part C** — exam-style Q&A.

---
# Part A · Tutor Review

> **Goal.** By the end of Part A you should be able to (a) state the RNN recurrence and the Jacobian-product argument for vanishing gradients, (b) explain why LSTM's *additive* cell state fixes it, and (c) articulate why self-attention's *parallel, direct* token-to-token connections replace recurrence altogether.

---
## §0 — Why Sequence Models?

Most previous weeks assumed fixed or grid-structured inputs:
- **MLP** — fixed-size vector input, no temporal structure.
- **CNN** — 2-D grid with local, translation-equivariant receptive fields.
- **GNN** — irregular relational structure, permutation-equivariant.

Sequence data (language, audio, time series, DNA) has a *different* inductive bias: inputs are **ordered**, **variable-length**, and each token depends on a potentially long history of earlier tokens.

| Architecture | Input processing | Memory mechanism | Parallelisable over time? |
|---|---|---|---|
| MLP | All at once (fixed size) | None | Yes |
| CNN | Local windows | Receptive field | Yes |
| **RNN** | One token at a time | Hidden state | **No** |
| **LSTM** | One token at a time | Gated cell state | **No** |
| **Transformer** | All at once | Attention weights | **Yes** |

The historical narrative for this tutorial: **RNN → LSTM → Transformer**. Each step *solves* a specific limitation of its predecessor.

---
## §1 — Vanilla RNN and the Vanishing Gradient

### 1.1 The recurrence

At every time step $t$, a vanilla RNN combines the current input $x_t$ with a summary $h_{t-1}$ of everything seen so far:

$$h_t = \tanh(W_{xh}\, x_t + W_{hh}\, h_{t-1} + b_h), \qquad y_t = W_{hy}\, h_t + b_y.$$

The hidden state $h_t \in \mathbb{R}^H$ is a **lossy compression** of the entire input history into a fixed-size vector. Everything the model knows about the past must fit in those $H$ dimensions.

### 1.2 The Jacobian-product argument

Consider the gradient of a loss at time step $T$ with respect to a hidden state at an *earlier* step $k$. By the chain rule,

$$\frac{\partial h_T}{\partial h_k} \;=\; \prod_{t=k+1}^{T} \frac{\partial h_t}{\partial h_{t-1}} \;=\; \prod_{t=k+1}^{T} W_{hh}^{\top}\,\mathrm{diag}\!\bigl(\tanh'(z_t)\bigr).$$

Two things multiply $T-k$ times:
- $|\tanh'(z)| \le 1$, and in practice $\ll 1$ when $z$ is not tiny;
- $\|W_{hh}\|$ is typically $<1$ after training.

So the product shrinks **exponentially** in $T-k$ → **vanishing gradients**. When $\|W_{hh}\| > 1$ it blows up instead → **exploding gradients**, which gradient clipping can patch but vanishing cannot.

**Bottom line.** A vanilla RNN *cannot learn long-range dependencies*: the supervision signal at step $T$ never reaches the weights at step $k$ when $T-k$ is large.

> We need a mechanism where gradients flow across many time steps **without multiplicative decay**. The key idea: an **additive** update path.

---
## §2 — LSTM: Gates and the Additive Cell State

LSTM introduces a second state vector, the **cell state** $c_t$, which acts as a conveyor belt. Information flows along it with only *additive* modifications, not multiplicative. Three **gates** — smooth switches with outputs in $[0,1]$ — control what gets written, what gets kept, and what gets read out.

| Symbol | Formula | Role |
|---|---|---|
| Forget gate $f_t$ | $\sigma(W_f[h_{t-1}, x_t] + b_f)$ | fraction of $c_{t-1}$ to **keep** |
| Input gate $i_t$ | $\sigma(W_i[h_{t-1}, x_t] + b_i)$ | fraction of new candidate to **write** |
| Candidate $\tilde c_t$ | $\tanh(W_c[h_{t-1}, x_t] + b_c)$ | proposed **new information** |
| **Cell update** | $c_t = f_t \odot c_{t-1} + i_t \odot \tilde c_t$ | *additive* — the key! |
| Output gate $o_t$ | $\sigma(W_o[h_{t-1}, x_t] + b_o)$ | fraction of cell state to **expose** |
| Hidden state | $h_t = o_t \odot \tanh(c_t)$ | output at time $t$ |

**Why this fixes vanishing gradients.** The gradient along the cell state between steps $k$ and $T$ is

$$\frac{\partial c_T}{\partial c_k} \;=\; \prod_{t=k+1}^{T} f_t.$$

If the network *learns* $f_t \approx 1$ (easy: just keep the forget-gate pre-activation large and positive), this product stays near $1$ — gradients flow across arbitrarily many time steps unchanged. A vanilla RNN has no weight configuration that makes the corresponding product identically $1$; LSTM does.

**Why sigmoid for gates and tanh for the candidate?** Sigmoids in $[0,1]$ act as soft switches (0 = block, 1 = pass). Tanh for $\tilde c_t$ is centred at $0$ so the cell state can both increase *and* decrease, giving symmetric gradient flow.

*(GRU is LSTM's sibling: 2 gates instead of 3, merged cell/hidden state. See the self-study notebook.)*

---
## §3 — From Recurrence to Attention

Even with LSTM, three problems remain:

1. **Sequential bottleneck.** $h_t$ still depends on $h_{t-1}$, so you **cannot parallelise** training over the time axis on a GPU — a length-$T$ sequence takes $T$ sequential steps no matter how much hardware you throw at it.
2. **Fixed-size memory.** Every past token must be crammed into a single hidden vector. Distant information competes for the same $H$ dimensions.
3. **Indirect paths.** Information from token $1$ can influence the output at token $T$ only by travelling through $T-1$ intermediate cells. Path length is $O(T)$.

**Self-attention** kills all three in one shot. Every token queries every other token *directly* using a learned similarity; the path length between any two positions is $O(1)$, and the whole operation is a few large matmuls that parallelise perfectly on a GPU.

### 3.1 Scaled dot-product attention

Given query, key and value matrices $Q, K, V$ (with $Q, K \in \mathbb{R}^{T\times d_k}$, $V \in \mathbb{R}^{T\times d_v}$),

$$\boxed{\;\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\!\left(\frac{Q K^{\top}}{\sqrt{d_k}}\right)V\;}$$

Reading the formula:
- $QK^{\top} \in \mathbb{R}^{T\times T}$ — raw similarity between every query and every key.
- **$\sqrt{d_k}$ scaling** — if $Q$ and $K$ have unit-variance entries, each dot product has variance $d_k$, so the standard deviation of the logits grows like $\sqrt{d_k}$. Without the scale, softmax saturates for large $d_k$ and gradients vanish through the softmax Jacobian. Dividing by $\sqrt{d_k}$ restores $O(1)$ logit variance. *(This is the subject of Exam Q1.)*
- **softmax over keys** (last axis) — each query's row of attention weights sums to $1$.
- **Masking** — to block some positions (e.g. future tokens in a causal LM), set their scores to $-\infty$ **before** softmax, so they become exactly $0$ **after** softmax. Setting them to $0$ after softmax would destroy the row-sum-to-one property.

### 3.2 Multi-head attention

A single attention head can only express one kind of similarity. **Multi-head** runs $h$ parallel attention computations on *different linear projections* of the same input, then concatenates the results and applies an output projection:

$$\mathrm{MultiHead}(Q, K, V) = \mathrm{Concat}(\mathrm{head}_1, \ldots, \mathrm{head}_h)\,W^O, \qquad \mathrm{head}_i = \mathrm{Attention}(QW_i^Q, KW_i^K, VW_i^V).$$

The standard convention is $d_k = d_v = d_{\text{model}} / h$, so the total FLOP count is *the same* as single-head attention with full $d_{\text{model}}$ — you trade width for multiple parallel views.

### 3.3 Positional encoding (brief)

Self-attention is **permutation-equivariant** over tokens (see Exam Q3): shuffle the input, the output is shuffled identically. That is a *disaster* for language — "dog bites man" ≠ "man bites dog". The fix is to **add** a position-dependent vector $\mathrm{PE}_t$ to each token embedding *before* feeding it into the attention stack. The classic sinusoidal scheme is

$$\mathrm{PE}_{t, 2i} = \sin\!\bigl(t / 10000^{2i/d_{\text{model}}}\bigr), \qquad \mathrm{PE}_{t, 2i+1} = \cos\!\bigl(t / 10000^{2i/d_{\text{model}}}\bigr).$$

The point is not the specific functions — it is that $\mathrm{PE}_t$ depends on position index only and is *not* permuted with the tokens, so it breaks the permutation symmetry.

---


# Part B · In-Class Exercise

> **Your job.** Fill in the `# TODO` blocks in the two tasks below. Each task has a collapsed **Solution** cell — try the task yourself first, then expand the solution to compare.
>
> The two models are:
> 1. **LSTM cell** — gating + additive cell-state update.
> 2. **Multi-head self-attention** — Q/K/V, scaled dot product with masking, head split/concat.

Before starting, run the imports.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
np.random.seed(42)

## Task B1 · Implement an `LSTMCell` from scratch

Recall the six LSTM equations. Gate pre-activations all use the same concatenated input $[h_{t-1}; x_t]$:

$$
\begin{aligned}
i_t &= \sigma(W_i\,[h_{t-1}; x_t] + b_i) && \text{(input gate)} \\
f_t &= \sigma(W_f\,[h_{t-1}; x_t] + b_f) && \text{(forget gate)} \\
\tilde c_t &= \tanh(W_c\,[h_{t-1}; x_t] + b_c) && \text{(candidate)} \\
o_t &= \sigma(W_o\,[h_{t-1}; x_t] + b_o) && \text{(output gate)} \\
c_t &= f_t \odot c_{t-1} + i_t \odot \tilde c_t && \text{(cell update — additive!)} \\
h_t &= o_t \odot \tanh(c_t) && \text{(hidden state)}
\end{aligned}
$$

**Implementation trick.** Concatenate the four weight matrices into one so you only do a *single* `nn.Linear` call per step. The output has shape $(B, 4H)$; split it into four equal chunks for $i, f, \tilde c, o$ pre-activations.

Fill in the six `# TODO` lines in the cell below.

In [ ]:
class LSTMCell(nn.Module):
    """From-scratch LSTM cell.

    Input at step t : x_t of shape (B, in_features)
    State           : (h_{t-1}, c_{t-1}), each of shape (B, hidden_size)
    Output          : (h_t, c_t)
    """

    def __init__(self, in_features: int, hidden_size: int):
        super().__init__()
        self.in_features = in_features
        self.hidden_size = hidden_size
        # One big linear layer for all four gates: input [h_{t-1}; x_t]  →  (i, f, g, o)
        self.gates = nn.Linear(in_features + hidden_size, 4 * hidden_size)

    def forward(self, x_t, state):
        h_prev, c_prev = state                         # each (B, H)

        # Concatenate previous hidden and current input:  (B, H + in_features)
        concat = torch.cat([h_prev, x_t], dim=1)

        # Single matmul → all four gate pre-activations stacked on the last axis
        gate_preacts = self.gates(concat)              # (B, 4H)
        i_pre, f_pre, g_pre, o_pre = gate_preacts.chunk(4, dim=1)   # each (B, H)

        # TODO 1 — input gate:     i_t = sigmoid(i_pre)
        i_t = ...

        # TODO 2 — forget gate:    f_t = sigmoid(f_pre)
        f_t = ...

        # TODO 3 — candidate cell: g_t = tanh(g_pre)
        g_t = ...

        # TODO 4 — output gate:    o_t = sigmoid(o_pre)
        o_t = ...

        # TODO 5 — additive cell-state update:  c_t = f_t * c_{t-1} + i_t * g_t
        c_t = ...

        # TODO 6 — hidden state:                h_t = o_t * tanh(c_t)
        h_t = ...

        return h_t, c_t

    def init_state(self, batch_size: int, device=None):
        h0 = torch.zeros(batch_size, self.hidden_size, device=device)
        c0 = torch.zeros(batch_size, self.hidden_size, device=device)
        return h0, c0

<details>
<summary><b>▸ Solution · Task B1</b> (click to expand)</summary>

```python
def forward(self, x_t, state):
    h_prev, c_prev = state
    concat = torch.cat([h_prev, x_t], dim=1)
    gate_preacts = self.gates(concat)
    i_pre, f_pre, g_pre, o_pre = gate_preacts.chunk(4, dim=1)

    i_t = torch.sigmoid(i_pre)          # TODO 1
    f_t = torch.sigmoid(f_pre)          # TODO 2
    g_t = torch.tanh(g_pre)             # TODO 3
    o_t = torch.sigmoid(o_pre)          # TODO 4
    c_t = f_t * c_prev + i_t * g_t      # TODO 5  — additive cell update
    h_t = o_t * torch.tanh(c_t)         # TODO 6
    return h_t, c_t
```

**Key points** (why, not just what):

- **Sigmoid on gates** gives outputs in $[0,1]$: the gate acts as a *soft on/off switch* that multiplies its target elementwise. A value of $0$ fully blocks, $1$ fully passes. Using tanh here would allow negative "anti-gates" which has no interpretation.
- **Tanh on the candidate** $\tilde c_t$ (and on $c_t$ before the output gate) keeps values symmetric in $[-1,1]$, so the cell state can both grow and shrink. This gives balanced gradient flow.
- **`c_t = f_t * c_prev + i_t * g_t` is the whole point of LSTM.** The update is *additive*: the Jacobian $\partial c_t / \partial c_{t-1} = \mathrm{diag}(f_t)$, so the long-range gradient is $\prod_t f_t$. If the network wants to remember something, it just needs $f_t \approx 1$ for those steps — and that is *learnable* because $f_t$ is a smooth function of the weights.
- **Single fused `nn.Linear`** for all four gates is both faster (one matmul instead of four) and identical to computing the four separately — `chunk(4, dim=1)` just slices the output tensor.
- **Why split with `chunk`?** `gate_preacts[:, :H]`, `[:, H:2H]`, etc. would work too — `chunk` is just cleaner and crash-loud if the dimensions are wrong.
</details>

### Demo · run the cell on a toy sequence

We roll the cell over a synthetic sequence of length $T = 8$ with batch size $B = 2$ and input dimension $4$, and print the hidden state at each step. This is just a sanity check — there is no training.

In [ ]:
# Tiny toy sequence to validate the LSTM cell
torch.manual_seed(0)

B, T, in_features, H = 2, 8, 4, 5
cell = LSTMCell(in_features=in_features, hidden_size=H)

x_seq = torch.randn(T, B, in_features)               # (T, B, in_features)
h, c = cell.init_state(batch_size=B)

print(f"Input sequence shape : {tuple(x_seq.shape)}  (T, B, in_features)")
print(f"Hidden state shape   : {tuple(h.shape)}  (B, H)")
print()
print("Per-step hidden state (batch item 0, first 3 dims):")
for t in range(T):
    h, c = cell(x_seq[t], (h, c))
    print(f"  t={t}:  h_t[0, :3] = {h[0, :3].detach().numpy().round(3)}   "
          f"||c_t||={c[0].norm().item():.3f}")

print()
print(f"Final hidden shape: {tuple(h.shape)}     final cell shape: {tuple(c.shape)}")

## Task B2 · Implement `MultiHeadSelfAttention` from scratch

We now build the other half of the tutorial: a **multi-head self-attention** block that takes an input $x$ of shape $(B, T, d_{\text{model}})$ and returns an output of the same shape.

Core formula (reprinted from Part A):

$$\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\!\left(\frac{QK^{\top}}{\sqrt{d_k}}\right)V$$

**Head splitting.** Instead of one attention with dimension $d_{\text{model}}$, we split into $h$ heads each of dimension $d_k = d_{\text{model}}/h$. Concretely:

1. Project $x$ to $Q, K, V$ each of shape $(B, T, d_{\text{model}})$ using three `nn.Linear` layers.
2. Reshape each to $(B, T, h, d_k)$, then **transpose to $(B, h, T, d_k)$** so that the attention matmul operates per head.
3. Run scaled dot-product attention — producing per-head outputs of shape $(B, h, T, d_k)$.
4. Transpose back and **reshape to $(B, T, d_{\text{model}})$** (this is the "concat heads" step — the contiguous memory layout *is* concatenation).
5. Apply a final output projection $W^O$.

**Masking** is optional. If provided, `mask` is broadcastable to the attention shape $(B, h, T, T)$ with `1` = keep and `0` = block. Masked positions get set to `-inf` **before** softmax so they become exactly $0$ **after** softmax.

Fill in the eight `# TODO` lines below.

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    """From-scratch multi-head self-attention block.

    Input  : x of shape (B, T, d_model)
    Output : (y, attn)  where
        y    : (B, T, d_model)
        attn : (B, num_heads, T, T)   per-head attention weights (for inspection)
    """

    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model   = d_model
        self.num_heads = num_heads
        self.d_k       = d_model // num_heads

        # Four linear projections. Each is (d_model → d_model) so it covers all heads at once.
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        B, T, _ = x.shape
        H, d_k  = self.num_heads, self.d_k

        # ── Step 1: project x to Q, K, V, each (B, T, d_model) ──────────────────
        # TODO 1 — Q = W_q(x)
        Q = ...
        # TODO 2 — K = W_k(x)
        K = ...
        # TODO 3 — V = W_v(x)
        V = ...

        # ── Step 2: split heads  (B, T, d_model) → (B, H, T, d_k) ───────────────
        # Hint: .view(B, T, H, d_k).transpose(1, 2)
        # TODO 4 — reshape Q, K, V onto H heads and put the head axis second
        Q = ...
        K = ...
        V = ...

        # ── Step 3: scaled dot-product attention ────────────────────────────────
        # TODO 5 — raw scores: matmul Q with K transposed on its last two axes,
        #          then divide by sqrt(d_k).  Expected shape (B, H, T, T).
        scores = ...

        # TODO 6 — apply mask (if provided) by setting masked positions to -inf
        #          BEFORE the softmax. Remember: mask==0 means "block".
        if mask is not None:
            scores = ...

        attn = F.softmax(scores, dim=-1)                 # (B, H, T, T)

        # TODO 7 — weighted sum of values:  attn @ V     → (B, H, T, d_k)
        out = ...

        # ── Step 4: concat heads  (B, H, T, d_k) → (B, T, d_model) ──────────────
        out = out.transpose(1, 2).contiguous().view(B, T, self.d_model)

        # ── Step 5: output projection ───────────────────────────────────────────
        # TODO 8 — apply self.W_o
        y = ...

        return y, attn

<details>
<summary><b>▸ Solution · Task B2</b> (click to expand)</summary>

```python
def forward(self, x, mask=None):
    B, T, _ = x.shape
    H, d_k  = self.num_heads, self.d_k

    # Step 1 — Q/K/V projections
    Q = self.W_q(x)                                              # TODO 1
    K = self.W_k(x)                                              # TODO 2
    V = self.W_v(x)                                              # TODO 3

    # Step 2 — split heads:  (B, T, d_model) → (B, H, T, d_k)
    Q = Q.view(B, T, H, d_k).transpose(1, 2)                     # TODO 4
    K = K.view(B, T, H, d_k).transpose(1, 2)
    V = V.view(B, T, H, d_k).transpose(1, 2)

    # Step 3 — scaled dot-product attention
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)   # TODO 5
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))        # TODO 6
    attn = F.softmax(scores, dim=-1)                                 # (B, H, T, T)
    out  = torch.matmul(attn, V)                                     # TODO 7 — (B, H, T, d_k)

    # Step 4 — concat heads back to (B, T, d_model)
    out = out.transpose(1, 2).contiguous().view(B, T, self.d_model)

    # Step 5 — output projection
    y = self.W_o(out)                                                # TODO 8
    return y, attn
```

**Key points** (why, not just what):

- **Why three *separate* linear layers for Q, K, V?** They encode three *different* roles (querying, being queried, being aggregated). Sharing a single projection would collapse those roles and hurt expressiveness. Each `nn.Linear(d_model, d_model)` projects *all* heads at once — head splitting happens purely via `view + transpose` and uses no extra parameters.
- **Why `view(B, T, H, d_k).transpose(1, 2)` and not `view(B, H, T, d_k)` directly?** Memory layout. The natural storage order from the linear layer is contiguous in $d_{\text{model}}$, so reshaping to $(B, T, H, d_k)$ puts the $H$ heads *next to each other* per token. Only after that reshape does the transpose swap $T$ and $H$ to land at $(B, H, T, d_k)$. Doing `view(B, H, T, d_k)` directly would interleave tokens across heads — wrong data in wrong slots.
- **Why divide by $\sqrt{d_k}$ and not $d_k$?** Because the *standard deviation* of a dot product of two i.i.d. unit-variance vectors scales like $\sqrt{d_k}$, not $d_k$. Dividing by $\sqrt{d_k}$ restores $O(1)$ logit variance — which is exactly the regime where softmax has well-conditioned gradients. Dividing by $d_k$ would over-flatten. *(Full derivation in Exam Q1.)*
- **Why `-inf` before softmax instead of `0` after?** Because `softmax(x) = 0` requires $x = -\infty$. Zeroing *after* softmax would break the row-sum-to-1 property and leak gradients back through the masked positions.
- **`.contiguous()` before the final `view`.** After `transpose`, the tensor's strides are non-contiguous; `view` requires contiguous memory. Forgetting `.contiguous()` is a classic bug — PyTorch will throw a runtime error.
- **Why an output projection $W^O$?** The concatenated heads live in the *same* $d_{\text{model}}$-dim space, but each block of $d_k$ coordinates corresponds to *one specific head's* output. $W^O$ lets the model learn an arbitrary linear mix across heads — without it, head $i$ would always write to coordinates $[i\cdot d_k : (i+1)\cdot d_k]$ and never interact with the others.
</details>

### Demo · run MHSA on a tiny random input

Batch size $2$, sequence length $4$, $d_{\text{model}} = 8$, $2$ heads (so $d_k = 4$). We run both (a) unmasked self-attention and (b) causal-masked self-attention, and verify that the causal version has exactly zero weight on future positions.

In [ ]:
torch.manual_seed(0)

B, T, d_model, num_heads = 2, 4, 8, 2
mhsa = MultiHeadSelfAttention(d_model=d_model, num_heads=num_heads)

x = torch.randn(B, T, d_model)
print(f"Input  x   : shape {tuple(x.shape)}")

# (a) Unmasked self-attention
y, attn = mhsa(x)
print(f"Output y   : shape {tuple(y.shape)}")
print(f"Attention  : shape {tuple(attn.shape)}  (B, H, T, T)")
print(f"Row sums   : {attn.sum(dim=-1)[0, 0].detach().numpy().round(4)}  (should all be 1.0)")

# (b) Causal mask: 1 on/below diagonal, 0 above. Shape (1, 1, T, T) broadcasts to (B, H, T, T).
causal = torch.tril(torch.ones(T, T)).view(1, 1, T, T)
y_c, attn_c = mhsa(x, mask=causal)

# Future positions must be exactly zero after softmax.
upper_mass = attn_c.triu(diagonal=1).abs().max().item()
print(f"\nCausal-mask max weight above diagonal: {upper_mass:.2e}  (must be 0)")

# Visualise batch 0, head 0
fig, axes = plt.subplots(1, 2, figsize=(8, 3.2))
for ax, W, title in [(axes[0], attn[0, 0],   "No mask"),
                     (axes[1], attn_c[0, 0], "Causal mask")]:
    im = ax.imshow(W.detach().numpy(), cmap="viridis", vmin=0, vmax=1)
    ax.set_title(title)
    ax.set_xlabel("key position")
    ax.set_ylabel("query position")
    fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

---
# Part C · Exam-Style Questions

> Three short-answer questions at medium-to-high difficulty, each with a collapsed **Model Answer**. Try them on paper first, then expand the answer to check your reasoning and pick up the "key points".

## Q1 · Why $\sqrt{d_k}$? (softmax / gradient flow)

Scaled dot-product attention computes

$$\mathrm{Attn}(Q, K, V) = \mathrm{softmax}\!\left(\frac{QK^{\top}}{\sqrt{d_k}}\right)V.$$

Suppose the entries of $Q$ and $K$ are i.i.d. with zero mean and unit variance.

**(a)** Show that each entry of $QK^{\top}$ has variance $d_k$.

**(b)** Explain what happens to the softmax output as $d_k \to \infty$ **without** the $\sqrt{d_k}$ scaling.

**(c)** Using the softmax Jacobian, explain why this is a problem for gradient-based training, and argue why dividing by $\sqrt{d_k}$ — as opposed to $d_k$ or $1$ — is the right fix.

<details>
<summary><b>▸ Model Answer · Q1</b></summary>

**(a)** Let $q, k \in \mathbb{R}^{d_k}$ be a query/key pair. Their inner product is $\langle q, k\rangle = \sum_{i=1}^{d_k} q_i k_i$. Each $q_i k_i$ has mean $\mathbb{E}[q_i]\mathbb{E}[k_i] = 0$ and variance $\mathrm{Var}(q_i)\mathrm{Var}(k_i) = 1$, and the terms are independent. Therefore $\mathrm{Var}(\langle q, k\rangle) = d_k$, and the standard deviation grows like $\sqrt{d_k}$.

**(b)** Without scaling, the logits fed into softmax have magnitude on the order of $\sqrt{d_k}$. For large $d_k$, the largest logit dominates and the softmax output concentrates on a single position — it approaches a one-hot distribution.

**(c)** The Jacobian of softmax is $J_{ij} = p_i(\delta_{ij} - p_j)$. When the output is near one-hot ($p \approx e_{i^\star}$), every entry of $J$ is near zero, so **gradients vanish into $Q$ and $K$**. The fix must keep the *variance* of the logits at $O(1)$, and that is exactly what dividing by $\sqrt{d_k}$ achieves. Dividing by $d_k$ instead would shrink the standard deviation to $1/\sqrt{d_k}\to 0$ — over-flattening the softmax and destroying the model's ability to *select* anything (the opposite failure mode).

**Key points:**
- Variance argument, not mean argument — softmax cares about *spread*.
- Saturating softmax ⇒ zero Jacobian ⇒ dead gradients, by exactly the same mechanism that kills deep sigmoid networks.
- $\sqrt{d_k}$ is the unique scale that preserves $O(1)$ logit variance.
</details>

---

## Q2 · RNN vs. self-attention: complexity, path length, parallelism

Consider processing a sequence of length $T$ with hidden / model dimension $d$.

**(a)** Give the time complexity of one forward pass through (i) a single LSTM layer and (ii) a single self-attention layer.

**(b)** Give the **maximum path length** between any two tokens in each model — i.e. the minimum number of layer operations a signal must traverse to connect token $i$ to token $j$.

**(c)** Training throughput matters as much as asymptotic cost. Which of the two architectures can be parallelised across the time dimension during training, and why can the other one fundamentally *not*?

**(d)** For what regime of $T$ and $d$ is self-attention cheaper than an LSTM in raw FLOPs, ignoring parallelism?

<details>
<summary><b>▸ Model Answer · Q2</b></summary>

**(a)** LSTM per step: four $O(d^2)$ gate matmuls (or one fused $O(d^2)$ matmul on concatenated input), repeated for $T$ steps → $O(Td^2)$. Self-attention: $QK^{\top}$ and the attention-weighted sum of $V$ cost $O(T^2 d)$; the $Q/K/V/O$ projections cost $O(Td^2)$. Total $O(T^2 d + Td^2)$.

**(b)** LSTM: $O(T)$ — information from token $1$ reaches token $T$ only by traversing $T-1$ recurrent steps, even if the forget gates let the gradient through. Self-attention: $O(1)$ — every token attends to every other token in a *single* layer.

**(c)** Self-attention can be fully parallelised over the time axis during training: the representation at position $t$ does **not** depend on the representation at position $t-1$ — all $T$ positions are computed from the same input tensor in a handful of large matmuls. The LSTM hidden / cell state satisfies $(h_t, c_t) = f(h_{t-1}, c_{t-1}, x_t)$, so computing step $t$ needs step $t-1$ to already be done. The recurrence is inherently sequential and cannot be unrolled on a GPU, no matter how many cores you have.

**(d)** Comparing the dominant terms, self-attention is cheaper in raw FLOPs when $T^2 d < T d^2$, i.e. when $T < d$. In practice modern Transformers run with $T \sim d$ or $T > d$, so Transformers win on **path length and parallelism**, not on asymptotic FLOPs — which is the important take-away.

**Key points:**
- Sequential dependence is the hard blocker for LSTM parallelism, not FLOPs.
- $O(1)$ path length is why attention models long-range dependencies so well.
- Asymptotic FLOPs and wall-clock throughput can disagree; GPU parallelism decides.
</details>

---

## Q3 · Permutation equivariance and positional encoding

Let $\mathrm{Attn}(X)$ denote self-attention applied to a sequence $X \in \mathbb{R}^{T\times d}$, with $Q = XW^Q$, $K = XW^K$, $V = XW^V$. Let $P \in \mathbb{R}^{T\times T}$ be any permutation matrix.

**(a)** Prove that $\mathrm{Attn}(PX) = P\,\mathrm{Attn}(X)$. (I.e., self-attention is **permutation-equivariant** over tokens.)

**(b)** A classmate argues:
> *"Since self-attention is permutation-equivariant, a Transformer cannot distinguish 'dog bites man' from 'man bites dog'. Adding positional encodings to the input embeddings fixes this because the positional vectors break the symmetry."*

Is the classmate's **conclusion** correct? Is the **reasoning** rigorous? If not, what is the subtle gap?

**(c)** In Week 5 (CNN) we relied on translation equivariance, and in Week 6 (GNN) we relied on permutation equivariance over nodes, as *desirable* inductive biases. For sequence modelling, is permutation equivariance a desirable inductive bias or an obstacle? Justify in one or two sentences.

<details>
<summary><b>▸ Model Answer · Q3</b></summary>

**(a)** With $X' = PX$ we get $Q' = PXW^Q = PQ$ and similarly $K' = PK$, $V' = PV$. Then
$$Q'{K'}^{\top} = PQK^{\top}P^{\top}.$$
Softmax is applied **row-wise**, and row permutation commutes with any row-wise function, so
$$\mathrm{softmax}\!\left(\tfrac{Q'{K'}^{\top}}{\sqrt{d_k}}\right) = P\,\mathrm{softmax}\!\left(\tfrac{QK^{\top}}{\sqrt{d_k}}\right)P^{\top}.$$
Multiplying by $V' = PV$ and using $P^{\top}P = I$:
$$\mathrm{Attn}(PX) = P\,\mathrm{softmax}(\cdots)P^{\top}PV = P\,\mathrm{softmax}(\cdots)V = P\,\mathrm{Attn}(X). \qquad \blacksquare$$

**(b)** **Conclusion correct, reasoning loose.** The conclusion that a vanilla Transformer cannot distinguish the two sentences is right. But "adding positional encodings breaks the symmetry" is not automatic. The symmetry is only broken because positional encodings are a **fixed reference frame**: $\mathrm{PE}$ depends on position index, not on token identity, and is **not** permuted along with the tokens. If you permuted $\mathrm{PE}$ together with $X$ (i.e., treated $\mathrm{PE}$ as part of the token features), you would end up with $\mathrm{Attn}(P(X+\mathrm{PE})) = P\,\mathrm{Attn}(X+\mathrm{PE})$ by part (a), and the problem would reappear. The gap in the classmate's argument is that they did not explain *why* $\mathrm{PE}$ is not permuted — that is the actual symmetry-breaking step.

**(c)** It is an **obstacle**. In sequence data (language, time series) the *order* is meaningful — "dog bites man" ≠ "man bites dog" — so a good sequence model must be position-*sensitive*. This is the opposite of GNNs, where node labels are arbitrary and permutation equivariance is exactly what we want, and different from CNNs, where translation equivariance holds because pixel coordinates lie in a metric space with meaningful shifts. Positional encoding exists precisely to cancel the symmetry that self-attention would otherwise enforce.

**Key points:**
- The proof uses only two facts: $P^{\top}P = I$ and row-wise commutation of softmax with $P$.
- "Breaks symmetry" needs to be spelled out — PE is a *fixed* frame, not a feature permuted with the tokens.
- CNN / GNN / Transformer inductive biases each match a specific symmetry group (translations, node permutations, *none* over sequence positions respectively).
</details>

---
## Summary

| Section | Key concept |
|---|---|
| **§1** | Vanilla RNN recurrence $h_t = \tanh(W_{xh}x_t + W_{hh}h_{t-1} + b)$; gradient is a Jacobian product $\Rightarrow$ vanishing / exploding |
| **§2** | LSTM gates + additive cell update $c_t = f_t\odot c_{t-1} + i_t\odot\tilde c_t$; long-range gradient $= \prod_t f_t$, learnable $\approx 1$ |
| **§3** | Self-attention $\mathrm{softmax}(QK^{\top}/\sqrt{d_k})V$; $O(1)$ path length, fully parallel; multi-head runs $h$ projections in parallel |
| **§3.3** | Positional encoding breaks the permutation symmetry of self-attention |

**Take-aways:**
- Vanilla RNNs fail on long range because of *multiplicative* gradient flow; LSTMs fix it with an *additive* cell state.
- Transformers go further: they drop recurrence entirely, trading sequential memory for direct all-pairs interactions and full GPU parallelism.
- Every architectural choice in Part B (four gates, $\sqrt{d_k}$ scaling, head splitting, output projection) maps directly onto a specific failure mode of the alternative — you should be able to *justify* each line, not just write it.
